In [1]:
import yaml
import json
import pickle
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import httpx
import os

import warnings
warnings.filterwarnings("ignore")

from prolific import ProlificAPIError, ProlificClient

pd.set_option('display.max_columns', None)

# config = yaml.safe_load(open("config.yaml", "r"))

client = ProlificClient(
    api_token=os.environ["prolific_key"])

# API Testing

In [2]:
ID = "6978ec7f65af76fdfe5640b8"

In [3]:
client.get_study(ID)

{'id': '6978ec7f65af76fdfe5640b8',
 'name': 'Human judgements of persuasiveness',
 'description': '<p>In this study, you will evaluate <strong>persuasiveness in debate transcripts</strong>. You will read transcripts from <strong>two separate debates</strong> on the same topic and judge which speaker is more persuasive.</p><p>Each debate features two participants, Speaker A and Speaker B. Although the debates involve different individuals, <strong>Speaker A represents the same ideological or political position in both debates</strong>.</p><p>Your task is to compare the persuasiveness of <strong>Speaker A across the two debates</strong>, focusing only on how effectively they argue their position.</p><p>There are <strong>no right or wrong answers</strong>—we are interested in your honest, subjective judgment based on the debate content. </p>',
 'total_available_places': 100,
 'reward': 60,
 'average_reward_per_hour': 1317.0,
 'external_study_url': 'https://tars.dcp.prod.aws.prolific.co/ac

In [4]:
responses = client.list_submissions(study=ID, page_size=1000)
print(len(responses["results"]))
print(json.dumps(responses['results'][0], indent=2))

107
{
  "id": "6978ed104fd1353ebbd8a84c",
  "participant_id": "5c93b062c9d93b0015fcbc02",
  "started_at": "2026-01-27T16:51:31.999000Z",
  "completed_at": "2026-01-27T16:52:29.973000Z",
  "is_complete": true,
  "time_taken": 57,
  "reward": 6000,
  "status": "APPROVED",
  "strata": {},
  "study_code": "NOCODE",
  "bonus_payments": [],
  "ip": "92.238.156.108",
  "return_requested": null,
  "has_siblings": false,
  "time_taken_under_auto_approval_threshold": true,
  "dynamic_payment_percentage": null
}


In [5]:
sub = client.get_submission(responses['results'][0]['id'])
sub

{'id': '6978ed104fd1353ebbd8a84c',
 'status': 'APPROVED',
 'participant': '5c93b062c9d93b0015fcbc02',
 'datetime_created': '2026-01-27T16:51:28Z',
 'started_at': '2026-01-27T16:51:31.999000Z',
 'ip': '92.238.156.108',
 'completed_at': '2026-01-27T16:52:29.973000Z',
 'payment_sent_at': None,
 'emailed_at': None,
 'timed_out_permenant': False,
 'started_index': 32,
 'is_approved': True,
 'entered_code': 'NOCODE',
 'reviewed_at': '2026-01-27T18:03:13.752000Z',
 'returned_at': None,
 'discount_amount': 0,
 'study': None,
 'time_remaining': 0,
 'study_url': 'https://tars.dcp.prod.aws.prolific.co/access-details/cfd86ff2-90d1-4e35-841e-5f124157e55d/allocate',
 'study_code': 'NOCODE',
 'study_id': '6978ec7f65af76fdfe5640b8',
 'reserve_until': None,
 'return_requested': None,
 'bonus_payments': [],
 'parent_study_id': None,
 '_links': {'self': {'href': 'https://api.prolific.com/api/v1/submissions/6978ed104fd1353ebbd8a84c/',
   'title': 'Current'},
  'related': [{'href': 'https://api.prolific.co

In [19]:
df = pd.read_csv("data/test-pilot-study-results-019c004e-c6f8-71dd-9c9c-86944a4f873e.csv")

In [20]:
df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_COMPARISONID,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,Annotator9_ID,Annotator9_Response,Annotator9_Timestamp,Annotator10_ID,Annotator10_Response,Annotator10_Timestamp,Annotator11_ID,Annotator11_Response,Annotator11_Timestamp,Annotator12_ID,Annotator12_Response,Annotator12_Timestamp,Annotator13_ID,Annotator13_Response,Annotator13_Timestamp,Annotator14_ID,Annotator14_Response,Annotator14_Timestamp,Annotator15_ID,Annotator15_Response,Annotator15_Timestamp,Annotator16_ID,Annotator16_Response,Annotator16_Timestamp,Annotator17_ID,Annotator17_Response,Annotator17_Timestamp,Annotator18_ID,Annotator18_Response,Annotator18_Timestamp,Annotator19_ID,Annotator19_Response,Annotator19_Timestamp,Annotator20_ID,Annotator20_Response,Annotator20_Timestamp
0,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Speaker A from Debate 2,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Speaker A from Debate 1,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Speaker A from Debate 1,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,Speaker A from Debate 1,2026-01-27T16:53:30.876Z,60d075b6af3d46debaf3cfd8,Both Speaker As were equally persuasive,2026-01-27T16:53:33.173Z,5dd0409d9ca8a312350a4118,Speaker A from Debate 1,2026-01-27T16:55:11.699Z,5d41925ef7b99500012b960e,Speaker A from Debate 1,2026-01-27T16:55:23.367Z,67696d49d386c9789b03b1da,Speaker A from Debate 1,2026-01-27T16:55:40.827Z,66f1c61208486d7c425dcefd,Speaker A from Debate 1,2026-01-27T16:55:45.647Z,5ef528c2a0603b16a793423d,Both Speaker As were equally persuasive,2026-01-27T16:55:47.568Z,5ede66237578b70c1548f73e,Speaker A from Debate 1,2026-01-27T16:56:37.851Z,5b68c9eb87af310001584803,Speaker A from Debate 2,2026-01-27T16:57:17.132Z,5b421b9ac2e3810001763252,Speaker A from Debate 2,2026-01-27T16:58:12.235Z,5f0d89b9547a8403103a0372,Speaker A from Debate 1,2026-01-27T16:58:34.338Z,60b542b8fbb1a6a2678f5054,Speaker A from Debate 1,2026-01-27T17:01:08.079Z,5755c957eb80c4000741a9ce,Speaker A from Debate 2,2026-01-27T17:01:52.818Z,66bbafb05de7e549455aa4ff,Speaker A from Debate 1,2026-01-27T17:05:00.034Z,6798ae13687f9a946efe51eb,Speaker A from Debate 1,2026-01-27T17:06:07.176Z,66a1561d05457dd7ec9cd3a8,Both Speaker As were equally persuasive,2026-01-27T17:10:56.125Z,5efc4644f6f5950008d892a2,Speaker A from Debate 2,2026-01-27T17:19:39.180Z
1,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,How confident are you in your choice?,Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Somewhat unsure,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Very confident,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Somewhat confident,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,Very confident,2026-01-27T16:53:30.876

In [21]:
df["Question"][0]

'Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?'

In [22]:
df[df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"]

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_COMPARISONID,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,Annotator9_ID,Annotator9_Response,Annotator9_Timestamp,Annotator10_ID,Annotator10_Response,Annotator10_Timestamp,Annotator11_ID,Annotator11_Response,Annotator11_Timestamp,Annotator12_ID,Annotator12_Response,Annotator12_Timestamp,Annotator13_ID,Annotator13_Response,Annotator13_Timestamp,Annotator14_ID,Annotator14_Response,Annotator14_Timestamp,Annotator15_ID,Annotator15_Response,Annotator15_Timestamp,Annotator16_ID,Annotator16_Response,Annotator16_Timestamp,Annotator17_ID,Annotator17_Response,Annotator17_Timestamp,Annotator18_ID,Annotator18_Response,Annotator18_Timestamp,Annotator19_ID,Annotator19_Response,Annotator19_Timestamp,Annotator20_ID,Annotator20_Response,Annotator20_Timestamp
0,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Speaker A from Debate 2,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Speaker A from Debate 1,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Speaker A from Debate 1,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,Speaker A from Debate 1,2026-01-27T16:53:30.876Z,60d075b6af3d46debaf3cfd8,Both Speaker As were equally persuasive,2026-01-27T16:53:33.173Z,5dd0409d9ca8a312350a4118,Speaker A from Debate 1,2026-01-27T16:55:11.699Z,5d41925ef7b99500012b960e,Speaker A from Debate 1,2026-01-27T16:55:23.367Z,67696d49d386c9789b03b1da,Speaker A from Debate 1,2026-01-27T16:55:40.827Z,66f1c61208486d7c425dcefd,Speaker A from Debate 1,2026-01-27T16:55:45.647Z,5ef528c2a0603b16a793423d,Both Speaker As were equally persuasive,2026-01-27T16:55:47.568Z,5ede66237578b70c1548f73e,Speaker A from Debate 1,2026-01-27T16:56:37.851Z,5b68c9eb87af310001584803,Speaker A from Debate 2,2026-01-27T16:57:17.132Z,5b421b9ac2e3810001763252,Speaker A from Debate 2,2026-01-27T16:58:12.235Z,5f0d89b9547a8403103a0372,Speaker A from Debate 1,2026-01-27T16:58:34.338Z,60b542b8fbb1a6a2678f5054,Speaker A from Debate 1,2026-01-27T17:01:08.079Z,5755c957eb80c4000741a9ce,Speaker A from Debate 2,2026-01-27T17:01:52.818Z,66bbafb05de7e549455aa4ff,Speaker A from Debate 1,2026-01-27T17:05:00.034Z,6798ae13687f9a946efe51eb,Speaker A from Debate 1,2026-01-27T17:06:07.176Z,66a1561d05457dd7ec9cd3a8,Both Speaker As were equally persuasive,2026-01-27T17:10:56.125Z,5efc4644f6f5950008d892a2,Speaker A from Debate 2,2026-01-27T17:19:39.180Z
2,019c004e-b479-7348-8a29-9781ff428810,6762026f-5d49-52d1-be4a-133965da5ff6,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""The NHS in the UK continues to pro...","Speaker B: ""So does that mean that the whole p...",right,human_7,gpt-4o_mas_rag,"('human_7', 'human_8')",159,"[{'speaker': 'gpt-4o_mas_rag', 'timestamp': '2...","('DL3D7S', '6XAVOS')",671d3690dc5df6d004cfb65e,Speaker A from Debate 1,2026-01-27T16:52:05.218Z,5bb62f1a3e969700019bd44d,Both Speaker As were equally persuasive,2026-01-27T16:53:05.218Z,6704539fe462b87182866f14,Speaker A from Debate 1,2026-01-27T16:53:12.361Z,5c03

In [23]:
annotator_cols = [col for col in df.columns if col.endswith('_Response')]
responses = df[annotator_cols].melt(value_name='response')['response']
response_counts = responses.value_counts()
print(response_counts)

response
Somewhat confident                         58
Speaker A from Debate 1                    51
Speaker A from Debate 2                    36
Very confident                             29
Both Speaker As were equally persuasive    13
Somewhat unsure                             7
Neutral                                     6
Name: count, dtype: int64


In [24]:
# annotator_cols = [col for col in df.columns if col.endswith('_Response')]

# df['sample1_count'] = (df[annotator_cols] == "Speaker A from Debate 1").sum(axis=1)
# df['sample2_count'] = (df[annotator_cols] == "Speaker A from Debate 2").sum(axis=1)

In [25]:
annotator_cols = [col for col in df.columns if col.endswith('_Response')]

# Only consider rows where the Question matches the target
target_rows = df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"

def count_sample1(row):
    return sum([
        1 if val == "Speaker A from Debate 1" else
        0.5 if val == "Both Speaker As were equally persuasive" else
        0
        for val in row
    ])

def count_sample2(row):
    return sum([
        1 if val == "Speaker A from Debate 2" else
        0.5 if val == "Both Speaker As were equally persuasive" else
        0
        for val in row
    ])

df.loc[target_rows, 'sample1_count'] = df.loc[target_rows, annotator_cols].apply(count_sample1, axis=1)
df.loc[target_rows, 'sample2_count'] = df.loc[target_rows, annotator_cols].apply(count_sample2, axis=1)

df

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_COMPARISONID,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,Annotator9_ID,Annotator9_Response,Annotator9_Timestamp,Annotator10_ID,Annotator10_Response,Annotator10_Timestamp,Annotator11_ID,Annotator11_Response,Annotator11_Timestamp,Annotator12_ID,Annotator12_Response,Annotator12_Timestamp,Annotator13_ID,Annotator13_Response,Annotator13_Timestamp,Annotator14_ID,Annotator14_Response,Annotator14_Timestamp,Annotator15_ID,Annotator15_Response,Annotator15_Timestamp,Annotator16_ID,Annotator16_Response,Annotator16_Timestamp,Annotator17_ID,Annotator17_Response,Annotator17_Timestamp,Annotator18_ID,Annotator18_Response,Annotator18_Timestamp,Annotator19_ID,Annotator19_Response,Annotator19_Timestamp,Annotator20_ID,Annotator20_Response,Annotator20_Timestamp,sample1_count,sample2_count
0,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Speaker A from Debate 2,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Speaker A from Debate 1,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Speaker A from Debate 1,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,Speaker A from Debate 1,2026-01-27T16:53:30.876Z,60d075b6af3d46debaf3cfd8,Both Speaker As were equally persuasive,2026-01-27T16:53:33.173Z,5dd0409d9ca8a312350a4118,Speaker A from Debate 1,2026-01-27T16:55:11.699Z,5d41925ef7b99500012b960e,Speaker A from Debate 1,2026-01-27T16:55:23.367Z,67696d49d386c9789b03b1da,Speaker A from Debate 1,2026-01-27T16:55:40.827Z,66f1c61208486d7c425dcefd,Speaker A from Debate 1,2026-01-27T16:55:45.647Z,5ef528c2a0603b16a793423d,Both Speaker As were equally persuasive,2026-01-27T16:55:47.568Z,5ede66237578b70c1548f73e,Speaker A from Debate 1,2026-01-27T16:56:37.851Z,5b68c9eb87af310001584803,Speaker A from Debate 2,2026-01-27T16:57:17.132Z,5b421b9ac2e3810001763252,Speaker A from Debate 2,2026-01-27T16:58:12.235Z,5f0d89b9547a8403103a0372,Speaker A from Debate 1,2026-01-27T16:58:34.338Z,60b542b8fbb1a6a2678f5054,Speaker A from Debate 1,2026-01-27T17:01:08.079Z,5755c957eb80c4000741a9ce,Speaker A from Debate 2,2026-01-27T17:01:52.818Z,66bbafb05de7e549455aa4ff,Speaker A from Debate 1,2026-01-27T17:05:00.034Z,6798ae13687f9a946efe51eb,Speaker A from Debate 1,2026-01-27T17:06:07.176Z,66a1561d05457dd7ec9cd3a8,Both Speaker As were equally persuasive,2026-01-27T17:10:56.125Z,5efc4644f6f5950008d892a2,Speaker A from Debate 2,2026-01-27T17:19:39.180Z,13.5,6.5
1,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,How confident are you in your choice?,Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Somewhat unsure,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Very confident,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Somewhat confident,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,V

In [26]:
# annotator_cols = [col for col in df.columns if col.endswith('_Response')]

# # Only consider rows where the Question matches the target
# target_rows = df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"

# df.loc[target_rows, 'sample1_count'] = (df.loc[target_rows, annotator_cols] == "Speaker A from Debate 1").sum(axis=1)
# df.loc[target_rows, 'sample2_count'] = (df.loc[target_rows, annotator_cols] == "Speaker A from Debate 2").sum(axis=1)

In [ ]:
likert_map = {
        "Very unsure": 1,
            "Somewhat unsure": 2,
                "Neutral": 3,
                    "Somewhat confident": 4,
                        "Very confident": 5
                        }


# Select only the rows for the confidence question
conf_rows = df["Question"] == "How confident are you in your choice?"

# Select annotator columns
annotator_cols = [col for col in df.columns if col.endswith('_Response')]

# Map likert responses to numbers
df_conf_numeric = df.loc[conf_rows, annotator_cols].replace(likert_map)

# Compute the mean for each row
df.loc[conf_rows, "average_confidence"] = df_conf_numeric.mean(axis=1)
df.loc[conf_rows, "std_confidence"] = df_conf_numeric.std(axis=1)

In [28]:
df

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_COMPARISONID,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,Annotator9_ID,Annotator9_Response,Annotator9_Timestamp,Annotator10_ID,Annotator10_Response,Annotator10_Timestamp,Annotator11_ID,Annotator11_Response,Annotator11_Timestamp,Annotator12_ID,Annotator12_Response,Annotator12_Timestamp,Annotator13_ID,Annotator13_Response,Annotator13_Timestamp,Annotator14_ID,Annotator14_Response,Annotator14_Timestamp,Annotator15_ID,Annotator15_Response,Annotator15_Timestamp,Annotator16_ID,Annotator16_Response,Annotator16_Timestamp,Annotator17_ID,Annotator17_Response,Annotator17_Timestamp,Annotator18_ID,Annotator18_Response,Annotator18_Timestamp,Annotator19_ID,Annotator19_Response,Annotator19_Timestamp,Annotator20_ID,Annotator20_Response,Annotator20_Timestamp,sample1_count,sample2_count,average_confidence,std_confidence
0,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Speaker A from Debate 2,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Speaker A from Debate 1,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Speaker A from Debate 1,2026-01-27T16:53:06.922Z,5bbc3d95f1f9ba000141d855,Speaker A from Debate 1,2026-01-27T16:53:30.876Z,60d075b6af3d46debaf3cfd8,Both Speaker As were equally persuasive,2026-01-27T16:53:33.173Z,5dd0409d9ca8a312350a4118,Speaker A from Debate 1,2026-01-27T16:55:11.699Z,5d41925ef7b99500012b960e,Speaker A from Debate 1,2026-01-27T16:55:23.367Z,67696d49d386c9789b03b1da,Speaker A from Debate 1,2026-01-27T16:55:40.827Z,66f1c61208486d7c425dcefd,Speaker A from Debate 1,2026-01-27T16:55:45.647Z,5ef528c2a0603b16a793423d,Both Speaker As were equally persuasive,2026-01-27T16:55:47.568Z,5ede66237578b70c1548f73e,Speaker A from Debate 1,2026-01-27T16:56:37.851Z,5b68c9eb87af310001584803,Speaker A from Debate 2,2026-01-27T16:57:17.132Z,5b421b9ac2e3810001763252,Speaker A from Debate 2,2026-01-27T16:58:12.235Z,5f0d89b9547a8403103a0372,Speaker A from Debate 1,2026-01-27T16:58:34.338Z,60b542b8fbb1a6a2678f5054,Speaker A from Debate 1,2026-01-27T17:01:08.079Z,5755c957eb80c4000741a9ce,Speaker A from Debate 2,2026-01-27T17:01:52.818Z,66bbafb05de7e549455aa4ff,Speaker A from Debate 1,2026-01-27T17:05:00.034Z,6798ae13687f9a946efe51eb,Speaker A from Debate 1,2026-01-27T17:06:07.176Z,66a1561d05457dd7ec9cd3a8,Both Speaker As were equally persuasive,2026-01-27T17:10:56.125Z,5efc4644f6f5950008d892a2,Speaker A from Debate 2,2026-01-27T17:19:39.180Z,13.5,6.5,NaN,NaN
1,019c004e-b441-77a8-b741-60a6e281c5a4,407a1775-fc8a-5da6-9bcd-c9879baf85cb,multiple_choice,How confident are you in your choice?,Debate the trade-offs of government interventi...,"Speaker B: ""The government should allow parent...","Speaker A: ""I don't trust that they will const...",left,human_7,human_12,"('human_7', 'human_8')",158,"[{'speaker': 'human_12', 'timestamp': '2026-01...","('1XHC4T', 'RSQGNQ')",5d4d9ea8c088850019ec5b32,Somewhat unsure,2026-01-27T16:50:50.367Z,5afbf02302b1b5000103e313,Very confident,2026-01-27T16:51:57.594Z,5c4592d7f608210001a4b0a8,Somewhat confident,2026-01-2

In [33]:
import krippendorff
import numpy as np

def compute_krippendorff_alpha_ordinal(df):
    response_map = {
        "Speaker A from Debate 1": 0,
        "Speaker A from Debate 2": 1,
        "Both Speaker As were equally persuasive": 2,
        np.nan: np.nan  # Keep missing as np.nan
    }
    target_question = "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"
    target_rows = df["Question"] == target_question
    annotator_cols = [col for col in df.columns if col.endswith('_Response')]
    # Replace and convert to float for np.nan support
    annotations = df.loc[target_rows, annotator_cols].replace(response_map).astype(float).T.values
    alpha = krippendorff.alpha(reliability_data=annotations, level_of_measurement='ordinal')
    print(f"Krippendorff's alpha (ordinal): {alpha:.3f}")
    return alpha

# Usage:
compute_krippendorff_alpha_ordinal(df)

Krippendorff's alpha (ordinal): 0.045


np.float64(0.0448304219600727)

In [34]:
import krippendorff
import numpy as np

def compute_krippendorff_alpha_nominal(df):
    response_map = {
        "Speaker A from Debate 1": 0,
        "Speaker A from Debate 2": 1,
        "Both Speaker As were equally persuasive": 2,
        np.nan: np.nan
    }
    target_question = "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"
    target_rows = df["Question"] == target_question
    annotator_cols = [col for col in df.columns if col.endswith('_Response')]
    annotations = df.loc[target_rows, annotator_cols].replace(response_map).astype(float).T.values
    alpha = krippendorff.alpha(reliability_data=annotations, level_of_measurement='nominal')
    print(f"Krippendorff's alpha (nominal): {alpha:.3f}")
    return alpha

# Usage:
compute_krippendorff_alpha_nominal(df)

Krippendorff's alpha (nominal): 0.080


np.float64(0.07977223138736633)

In [29]:
participant_IDs = [col for col in df.columns if col.startswith("Annotator") and col.endswith("_ID")]
unique_participant_ids = pd.unique(df[participant_IDs].values.ravel())
unique_participant_ids = [pid for pid in unique_participant_ids if pd.notnull(pid)]
unique_participant_ids

['5d4d9ea8c088850019ec5b32',
 '5afbf02302b1b5000103e313',
 '5c4592d7f608210001a4b0a8',
 '5bbc3d95f1f9ba000141d855',
 '60d075b6af3d46debaf3cfd8',
 '5dd0409d9ca8a312350a4118',
 '5d41925ef7b99500012b960e',
 '67696d49d386c9789b03b1da',
 '66f1c61208486d7c425dcefd',
 '5ef528c2a0603b16a793423d',
 '5ede66237578b70c1548f73e',
 '5b68c9eb87af310001584803',
 '5b421b9ac2e3810001763252',
 '5f0d89b9547a8403103a0372',
 '60b542b8fbb1a6a2678f5054',
 '5755c957eb80c4000741a9ce',
 '66bbafb05de7e549455aa4ff',
 '6798ae13687f9a946efe51eb',
 '66a1561d05457dd7ec9cd3a8',
 '5efc4644f6f5950008d892a2',
 '671d3690dc5df6d004cfb65e',
 '5bb62f1a3e969700019bd44d',
 '6704539fe462b87182866f14',
 '5c03e86d55555b0001caf233',
 '5ac4b010f69e940001d97645',
 '54aefa06fdf99b09c01b37f4',
 '5a5a6ed7f6c517000195029e',
 '65537e14439274fe5036b4d2',
 '668dc7b684452eaca13eb60d',
 '56a757a95ad8d9000c970c76',
 '629dd4199343d7710fcdb39c',
 '5a0f34de9b760100013a7afd',
 '6125646031b73d8171421e7c',
 '5f0cf299c6db5770064402e1',
 '600742135622

In [22]:
client.bulk_approve_submissions(study_id=ID, participant_ids=unique_participant_ids)

'The request to bulk approve has been made successfully.'